# Step 8f — Hybrid Architecture

Modular DeBERTa-based difficulty classifier with optional scalar mix, prompt features, static features, CORN ordinal head, and DANN domain adversarial training.

Run the notebook top-to-bottom on Colab/A100. Edit only the `RUN_ARGS` cell near the bottom to choose the variant.


In [1]:
!pip install -q transformers accelerate sentencepiece scipy


In [2]:
# Mount Drive + set PROJECT_ROOT
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/text-difficulty-classification'
except Exception:
    PROJECT_ROOT = os.path.abspath('.')
os.environ['PROJECT_ROOT'] = PROJECT_ROOT
print('PROJECT_ROOT =', PROJECT_ROOT)


Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/text-difficulty-classification


In [3]:
import argparse
import json
import os
import pickle
import sys
import time
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

PROJECT_ROOT = os.environ['PROJECT_ROOT']
MULTI_DIR    = os.path.join(PROJECT_ROOT, 'outputs', 'multi_corpus')
PROMPT_DIR   = os.path.join(PROJECT_ROOT, 'outputs', 'prompt_metrics')
GENERIC_DIR  = os.path.join(PROJECT_ROOT, 'outputs', 'static_metrics_generic')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'outputs', 'models_multi')
HYBRID_DIR   = os.path.join(MODELS_DIR, 'hybrid')
os.makedirs(HYBRID_DIR, exist_ok=True)

LABEL_MAP = {'elementary': 0, 'middle': 1, 'high': 2}
LABEL_NAMES = ['elementary', 'middle', 'high']
NUM_CLASSES = 3


def _counts(y):
    return np.bincount(np.asarray(y, dtype=np.int64), minlength=NUM_CLASSES).tolist()


def _validate_split_df(df, split):
    missing = [c for c in ('full_text', 'education_level', 'source_dataset')
               if c not in df.columns]
    if missing:
        sys.exit(f"[step8f] {split}: missing required columns {missing}")
    bad = df[~df['education_level'].isin(LABEL_MAP)]
    if len(bad):
        sys.exit(f"[step8f] {split}: unknown labels: "
                 f"{bad['education_level'].dropna().unique().tolist()[:10]}")
    if df['full_text'].isna().any():
        sys.exit(f"[step8f] {split}: full_text has missing values")
    print(f"[step8f] {split}: rows={len(df)} "
          f"labels={df['education_level'].map(LABEL_MAP).pipe(_counts)} "
          f"sources={df['source_dataset'].value_counts().to_dict()}")
    return df


def _check_feature_rows(split, name, arr, n_rows):
    if arr is None:
        sys.exit(f"[step8f] missing {name} features for {split}")
    if len(arr) != n_rows:
        sys.exit(f"[step8f] {split}: {name} row mismatch. "
                 f"features={len(arr)} split_rows={n_rows}. "
                 "Regenerate this feature file from the same Step 2c outputs.")
    if not np.isfinite(arr).all():
        sys.exit(f"[step8f] {split}: {name} features contain NaN/inf")


def _assert_finite_model(model, where):
    bad = []
    for name, param in model.named_parameters():
        if not torch.isfinite(param).all():
            bad.append(name)
            if len(bad) >= 5:
                break
    if bad:
        raise RuntimeError(f"[step8f] non-finite model weights {where}: {bad}")


def _assert_finite_grads(model, where):
    bad = []
    for name, param in model.named_parameters():
        if param.grad is not None and not torch.isfinite(param.grad).all():
            bad.append(name)
            if len(bad) >= 5:
                break
    if bad:
        raise RuntimeError(f"[step8f] non-finite gradients {where}: {bad}")


In [4]:
class ScalarMix(nn.Module):
    """Weighted sum across encoder hidden layers (ELMo / Gombert 2024 BEA).
    Stable softmax over learnable weights, plus a learnable scalar gamma."""

    def __init__(self, num_layers):
        super().__init__()
        self.weights = nn.Parameter(torch.zeros(num_layers))
        self.gamma = nn.Parameter(torch.ones(1))

    def forward(self, hidden_states):
        # hidden_states: tuple/list of L tensors, each (B, T, H)
        w = F.softmax(self.weights, dim=0)
        stacked = torch.stack(hidden_states, dim=0)        # (L, B, T, H)
        mixed = (w.view(-1, 1, 1, 1) * stacked).sum(dim=0)  # (B, T, H)
        return self.gamma * mixed


In [5]:
class _GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = float(lambd)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


In [6]:
def grad_reverse(x, lambd):
    """Gradient reversal layer (Ganin & Lempitsky 2016)."""
    return _GradReverse.apply(x, lambd)


In [7]:
def corn_label(y, num_classes):
    """y (B,) int → (B, K-1) binary targets where col k = (y > k)."""
    return torch.stack([(y > k).float() for k in range(num_classes - 1)], dim=1)


In [8]:
def corn_mask(y, num_classes):
    """Conditional mask: col k contributes only when sample is at-or-above
    rank k (i.e. y >= k). Shape (B, K-1)."""
    return torch.stack([(y >= k).float() for k in range(num_classes - 1)], dim=1)


In [9]:
def corn_loss(logits, y, num_classes):
    """CORN (Cao et al. 2020) — rank-consistent ordinal regression.
    `logits` (B, K-1) are scored independently with BCE, but only on samples
    where the rank condition y >= k holds. Avoids rank-inconsistency that
    plagues simple cumulative-link models."""
    targets = corn_label(y, num_classes)
    mask    = corn_mask(y, num_classes)
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    return (bce * mask).sum() / mask.sum().clamp(min=1)


In [10]:
def corn_predict(logits):
    """Sequential decoding: chain conditional sigmoids to get unconditional
    P(y > k), pick predicted class as count of probs > 0.5."""
    probs = torch.sigmoid(logits)
    cumprobs = torch.cumprod(probs, dim=1)
    return (cumprobs > 0.5).sum(dim=1)


In [11]:
class HybridModel(nn.Module):
    """Modular fusion model. Each branch toggleable via __init__ flags."""

    def __init__(self, encoder_name, num_classes,
                 use_scalar_mix=False, use_prompts=False, use_static=False,
                 prompt_dim=63, static_dim=26,
                 hidden_dim=256, head_type='softmax',
                 use_dann=False, num_domains=1, dropout=0.1):
        super().__init__()
        from transformers import AutoModel
        self.encoder = AutoModel.from_pretrained(
            encoder_name, output_hidden_states=use_scalar_mix)
        enc_dim = self.encoder.config.hidden_size

        self.use_scalar_mix = use_scalar_mix
        if use_scalar_mix:
            # encoder.config.num_hidden_layers excludes embedding layer; transformer
            # returns embeddings + N hidden layers = N+1 tensors in hidden_states
            self.scalar_mix = ScalarMix(self.encoder.config.num_hidden_layers + 1)

        self.use_prompts = use_prompts
        self.use_static  = use_static
        self.head_type   = head_type
        self.use_dann    = use_dann
        self.dropout     = nn.Dropout(dropout)

        fusion_in = enc_dim
        if use_prompts: fusion_in += prompt_dim
        if use_static:  fusion_in += static_dim

        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Difficulty head
        if head_type == 'softmax':
            self.diff_head = nn.Linear(hidden_dim, num_classes)
        elif head_type == 'corn':
            self.diff_head = nn.Linear(hidden_dim, num_classes - 1)
        else:
            raise ValueError(f"unknown head_type {head_type}")

        # Domain head (only used if --use_dann)
        if use_dann:
            self.domain_head = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, num_domains),
            )

    def _pool(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        if self.use_scalar_mix:
            mixed = self.scalar_mix(out.hidden_states)  # (B, T, H)
            # Mean-pool over non-pad tokens (more robust than CLS for ordinal eval)
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (mixed * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            # Use [CLS] from last hidden state
            pooled = out.last_hidden_state[:, 0]
        return pooled

    def forward(self, input_ids, attention_mask,
                prompt_feats=None, static_feats=None,
                dann_lambda: Optional[float] = None):
        pooled = self._pool(input_ids, attention_mask)

        parts = [pooled]
        if self.use_prompts:
            assert prompt_feats is not None
            parts.append(prompt_feats)
        if self.use_static:
            assert static_feats is not None
            parts.append(static_feats)
        x = torch.cat(parts, dim=-1) if len(parts) > 1 else pooled

        h = self.fusion(self.dropout(x))
        diff_logits = self.diff_head(h)

        domain_logits = None
        if self.use_dann and dann_lambda is not None:
            h_rev = grad_reverse(h, dann_lambda)
            domain_logits = self.domain_head(h_rev)

        return diff_logits, domain_logits


In [12]:
# ---------- data ----------


In [13]:
class FusionDataset(Dataset):
    def __init__(self, texts, labels, source_ids, prompt_feats=None,
                 static_feats=None, tokenizer=None, max_len=512):
        self.texts = texts
        self.labels = labels
        self.source_ids = source_ids
        self.prompt_feats = prompt_feats
        self.static_feats = static_feats
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, max_length=self.max_len,
                       padding='max_length', return_tensors='pt')
        item = {
            'input_ids':      enc['input_ids'][0],
            'attention_mask': enc['attention_mask'][0],
            'labels':         torch.tensor(self.labels[i], dtype=torch.long),
            'source_id':      torch.tensor(self.source_ids[i], dtype=torch.long),
        }
        if self.prompt_feats is not None:
            item['prompt_feats'] = torch.tensor(self.prompt_feats[i], dtype=torch.float32)
        if self.static_feats is not None:
            item['static_feats'] = torch.tensor(self.static_feats[i], dtype=torch.float32)
        return item


In [14]:
def _load_split(split):
    p = os.path.join(MULTI_DIR, f'{split}.csv')
    if not os.path.exists(p):
        return None
    return pd.read_csv(p)


In [15]:
def _load_prompt_feats(split, model_key):
    p = os.path.join(PROMPT_DIR, f'{split}_prompt_metrics_{model_key}__multi.csv')
    if not os.path.exists(p):
        return None
    df = pd.read_csv(p)
    cols = sorted([c for c in df.columns if c.startswith('prompt_')],
                  key=lambda s: int(s.split('_')[1]))
    if not cols:
        sys.exit(f"[step8f] {split}: no prompt_* columns in {p}")
    arr = df[cols].astype(np.float32).to_numpy()
    # Hedge -1 -> col mean of valid rows.
    for j in range(arr.shape[1]):
        col = arr[:, j]
        if (col < 0).any():
            mean = col[col >= 0].mean() if (col >= 0).any() else 0.0
            col[col < 0] = mean
    return arr


In [16]:
def _load_static_feats(split):
    p = os.path.join(GENERIC_DIR, f'{split}_static_generic.csv')
    if not os.path.exists(p):
        return None
    df = pd.read_csv(p)
    cols = sorted([c for c in df.columns if c.startswith('gen_')])
    if not cols:
        sys.exit(f"[step8f] {split}: no gen_* columns in {p}")
    arr = df[cols].astype(np.float32).to_numpy()
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return arr, cols


In [17]:
def _list_ood():
    return sorted(f[len('ood_'):-len('.csv')]
                  for f in os.listdir(MULTI_DIR)
                  if f.startswith('ood_') and f.endswith('.csv'))


In [18]:
def _build_source_index(*split_dfs):
    """Map each unique source_dataset string → integer id, used by domain head."""
    sources = set()
    for df in split_dfs:
        if df is not None:
            sources.update(df['source_dataset'].astype(str).tolist())
    return {s: i for i, s in enumerate(sorted(sources))}


In [19]:
# ---------- training ----------


In [20]:
def train_one_epoch(model, loader, optim, sched, device, args, num_train_steps,
                    global_step_state):
    model.train()
    total_loss, total_diff, total_dom = 0.0, 0.0, 0.0
    n = 0
    for step, batch in enumerate(tqdm(loader, desc='train', leave=False)):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        y    = batch['labels'].to(device)
        prompt = batch.get('prompt_feats').to(device) if 'prompt_feats' in batch else None
        static = batch.get('static_feats').to(device) if 'static_feats' in batch else None
        sid    = batch.get('source_id').to(device)    if args.use_dann else None

        # DANN λ schedule — linear ramp from 0 to args.dann_lambda.
        lambd = None
        if args.use_dann:
            warm = max(1, int(args.dann_warmup_frac * num_train_steps))
            t = global_step_state['step']
            lambd = min(args.dann_lambda, args.dann_lambda * t / warm)

        diff_logits, dom_logits = model(ids, mask, prompt_feats=prompt,
                                        static_feats=static, dann_lambda=lambd)
        if not torch.isfinite(diff_logits).all():
            raise RuntimeError(f"[step8f] non-finite difficulty logits at step={step}")

        if args.head == 'softmax':
            l_diff = F.cross_entropy(diff_logits, y)
        else:
            l_diff = corn_loss(diff_logits, y, NUM_CLASSES)
        if not torch.isfinite(l_diff):
            raise RuntimeError(f"[step8f] non-finite difficulty loss at step={step}")

        loss = l_diff
        if args.use_dann and dom_logits is not None and sid is not None:
            if not torch.isfinite(dom_logits).all():
                raise RuntimeError(f"[step8f] non-finite domain logits at step={step}")
            l_dom = F.cross_entropy(dom_logits, sid)
            if not torch.isfinite(l_dom):
                raise RuntimeError(f"[step8f] non-finite domain loss at step={step}")
            loss = loss + l_dom    # gradient reversal already applied internally
        if not torch.isfinite(loss):
            raise RuntimeError(f"[step8f] non-finite total loss at step={step}")

        optim.zero_grad(set_to_none=True)
        loss.backward()
        _assert_finite_grads(model, f"step={step}")
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip_grad)
        optim.step()
        sched.step()
        _assert_finite_model(model, f"after step={step}")

        total_loss += loss.item() * y.size(0)
        total_diff += l_diff.item() * y.size(0)
        if args.use_dann and dom_logits is not None:
            total_dom += l_dom.item() * y.size(0)
        n += y.size(0)
        global_step_state['step'] += 1
    return total_loss / max(n, 1), total_diff / max(n, 1), total_dom / max(n, 1)


In [21]:
@torch.no_grad()
def evaluate(model, loader, device, head_type):
    model.eval()
    preds, ys = [], []
    for batch in loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        prompt = batch.get('prompt_feats').to(device) if 'prompt_feats' in batch else None
        static = batch.get('static_feats').to(device) if 'static_feats' in batch else None
        diff_logits, _ = model(ids, mask, prompt_feats=prompt, static_feats=static)
        if not torch.isfinite(diff_logits).all():
            raise RuntimeError("[step8f] non-finite difficulty logits during evaluation")
        if head_type == 'softmax':
            p = diff_logits.argmax(-1)
        else:
            p = corn_predict(diff_logits)
        preds.append(p.cpu().numpy())
        ys.append(batch['labels'].numpy())
    y = np.concatenate(ys); p = np.concatenate(preds)
    return {
        'f1_macro':    float(f1_score(y, p, average='macro', labels=[0, 1, 2], zero_division=0)),
        'f1_per_class': f1_score(y, p, average=None, labels=[0, 1, 2], zero_division=0).tolist(),
        'accuracy':    float(accuracy_score(y, p)),
        'mae':         float(mean_absolute_error(y, p)),                # ordinal-aware
        'qwk':         float(cohen_kappa_score(y, p, weights='quadratic')),
        'n':           int(len(y)),
        'pred_counts': _counts(p),
        'y_pred':      p.tolist(),
    }


In [22]:
# ---------- result append ----------


In [23]:
def _append_result(rec):
    out = os.path.join(MODELS_DIR, 'results.json')
    existing = json.load(open(out)) if os.path.exists(out) else []
    existing = [r for r in existing
                if not (r.get('feature_set') == rec['feature_set']
                        and r.get('classifier') == rec['classifier']
                        and r.get('seed') == rec.get('seed'))]
    existing.append(rec)
    json.dump(existing, open(out, 'w'), indent=2, default=str)


In [24]:
def _config_tag(args):
    parts = [args.encoder.split('/')[-1]]
    if args.use_scalar_mix: parts.append('mix')
    if args.use_prompts:    parts.append(f'prom-{args.prompt_model}')
    if args.use_static:     parts.append('stat')
    parts.append(args.head)
    if args.use_dann: parts.append(f'dann{args.dann_lambda}')
    return '__'.join(parts)


In [25]:
# ---------- main ----------


In [26]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--encoder', default='microsoft/deberta-v3-base')
    ap.add_argument('--use_scalar_mix', action='store_true')
    ap.add_argument('--use_prompts',    action='store_true')
    ap.add_argument('--prompt_model',   default='qwen2.5-7b',
                    help='Suffix tag in <split>_prompt_metrics_<model>__multi.csv')
    ap.add_argument('--use_static',     action='store_true')
    ap.add_argument('--head',           choices=['softmax', 'corn'], default='softmax')
    ap.add_argument('--use_dann',       action='store_true')
    ap.add_argument('--dann_lambda',    type=float, default=1.0)
    ap.add_argument('--dann_warmup_frac', type=float, default=0.5)

    ap.add_argument('--epochs',     type=int, default=3)
    ap.add_argument('--lr',         type=float, default=2e-5)
    ap.add_argument('--head_lr',    type=float, default=1e-4,
                    help='LR for fusion, difficulty, domain, and scalar-mix params.')
    ap.add_argument('--batch_size', type=int, default=16)
    ap.add_argument('--max_len',    type=int, default=512)
    ap.add_argument('--seed',       type=int, default=42)
    ap.add_argument('--hidden_dim', type=int, default=256)
    ap.add_argument('--clip_grad',  type=float, default=1.0)

    if any(a.endswith('.json') or a.startswith('-f') for a in sys.argv[1:]):
        args = ap.parse_args([])
    else:
        args = ap.parse_args()

    print(f"[step8f] config={_config_tag(args)} seed={args.seed}")

    if not torch.cuda.is_available():
        sys.exit("[step8f] needs GPU.")
    device = 'cuda'
    torch.manual_seed(args.seed); np.random.seed(args.seed)
    torch.set_default_dtype(torch.float32)

    # ---- splits ----
    train_df = _load_split('train')
    val_df   = _load_split('val')
    test_df  = _load_split('test')
    if any(d is None for d in (train_df, val_df, test_df)):
        sys.exit("[step8f] missing train/val/test in outputs/multi_corpus/.")
    train_df = _validate_split_df(train_df, 'train')
    val_df   = _validate_split_df(val_df,   'val')
    test_df  = _validate_split_df(test_df,  'test')

    ood_corpora = _list_ood()
    ood_dfs = {c: _validate_split_df(_load_split(f'ood_{c}'), f'ood_{c}')
               for c in ood_corpora}

    src2id = _build_source_index(train_df, val_df, test_df, *ood_dfs.values())
    num_domains = len(src2id)
    print(f"[step8f] sources: {src2id}")

    # ---- prompts (optional) ----
    train_prompts = val_prompts = test_prompts = None
    ood_prompts = {}
    if args.use_prompts:
        train_prompts = _load_prompt_feats('train', args.prompt_model)
        val_prompts   = _load_prompt_feats('val',   args.prompt_model)
        test_prompts  = _load_prompt_feats('test',  args.prompt_model)
        for c in ood_corpora:
            ood_prompts[c] = _load_prompt_feats(f'ood_{c}', args.prompt_model)
        if any(x is None for x in (train_prompts, val_prompts, test_prompts)):
            sys.exit(f"[step8f] missing prompt CSV for {args.prompt_model}. "
                     "Run Step 5c first.")
        _check_feature_rows('train', 'prompt', train_prompts, len(train_df))
        _check_feature_rows('val', 'prompt', val_prompts, len(val_df))
        _check_feature_rows('test', 'prompt', test_prompts, len(test_df))
        for c in ood_corpora:
            _check_feature_rows(f'ood_{c}', 'prompt', ood_prompts[c], len(ood_dfs[c]))
        prompt_dim = train_prompts.shape[1]
    else:
        prompt_dim = 0

    # ---- static (optional, scaled) ----
    train_static = val_static = test_static = None
    ood_static = {}
    static_dim = 0
    if args.use_static:
        train_static_loaded = _load_static_feats('train')
        val_static_loaded   = _load_static_feats('val')
        test_static_loaded  = _load_static_feats('test')
        if any(x is None for x in (train_static_loaded, val_static_loaded, test_static_loaded)):
            sys.exit("[step8f] missing static feature CSV. Run Step 3b first.")
        train_static_arr, _ = train_static_loaded
        val_static_arr, _   = val_static_loaded
        test_static_arr, _  = test_static_loaded
        _check_feature_rows('train', 'static', train_static_arr, len(train_df))
        _check_feature_rows('val', 'static', val_static_arr, len(val_df))
        _check_feature_rows('test', 'static', test_static_arr, len(test_df))
        scaler = StandardScaler().fit(train_static_arr)
        train_static = scaler.transform(train_static_arr).astype(np.float32)
        val_static   = scaler.transform(val_static_arr).astype(np.float32)
        test_static  = scaler.transform(test_static_arr).astype(np.float32)
        for c in ood_corpora:
            loaded = _load_static_feats(f'ood_{c}')
            if loaded is None:
                sys.exit(f"[step8f] missing static feature CSV for ood_{c}. Run Step 3b first.")
            arr, _ = loaded
            _check_feature_rows(f'ood_{c}', 'static', arr, len(ood_dfs[c]))
            ood_static[c] = scaler.transform(arr).astype(np.float32)
        static_dim = train_static.shape[1]

    # ---- tokenizer + model ----
    from transformers import AutoTokenizer, get_linear_schedule_with_warmup
    tokenizer = AutoTokenizer.from_pretrained(args.encoder)
    model = HybridModel(
        encoder_name   = args.encoder,
        num_classes    = NUM_CLASSES,
        use_scalar_mix = args.use_scalar_mix,
        use_prompts    = args.use_prompts,
        use_static     = args.use_static,
        prompt_dim     = prompt_dim,
        static_dim     = static_dim,
        hidden_dim     = args.hidden_dim,
        head_type      = args.head,
        use_dann       = args.use_dann,
        num_domains    = num_domains,
    )
    model = model.float().to(device)
    _assert_finite_model(model, "after load")

    def make_loader(df, prompts, static, shuffle):
        labels = df['education_level'].map(LABEL_MAP).astype(np.int64).to_numpy()
        sids   = df['source_dataset'].astype(str).map(src2id).to_numpy()
        ds = FusionDataset(
            df['full_text'].astype(str).tolist(), labels, sids,
            prompt_feats=prompts, static_feats=static,
            tokenizer=tokenizer, max_len=args.max_len)
        return DataLoader(ds, batch_size=args.batch_size, shuffle=shuffle, num_workers=2)

    train_loader = make_loader(train_df, train_prompts, train_static, True)
    val_loader   = make_loader(val_df,   val_prompts,   val_static,   False)
    test_loader  = make_loader(test_df,  test_prompts,  test_static,  False)

    # ---- optim ----
    from torch.optim import AdamW
    encoder_params, head_params = [], []
    for name, param in model.named_parameters():
        if name.startswith('encoder.'):
            encoder_params.append(param)
        else:
            head_params.append(param)
    optim = AdamW([
        {'params': encoder_params, 'lr': args.lr},
        {'params': head_params, 'lr': args.head_lr},
    ], weight_decay=0.01, eps=1e-6)
    total_steps = len(train_loader) * args.epochs
    sched = get_linear_schedule_with_warmup(
        optim, num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps)

    # ---- train ----
    t0 = time.time()
    best_val_f1 = -1.0
    best_state = None
    global_step_state = {'step': 0}
    for epoch in range(args.epochs):
        avg_loss, avg_diff, avg_dom = train_one_epoch(
            model, train_loader, optim, sched, device, args,
            num_train_steps=total_steps, global_step_state=global_step_state)
        val_metrics = evaluate(model, val_loader, device, args.head)
        print(f"[step8f] epoch {epoch+1}: loss={avg_loss:.3f} (diff={avg_diff:.3f} "
              f"dom={avg_dom:.3f}) val_f1={val_metrics['f1_macro']:.3f} "
              f"qwk={val_metrics['qwk']:.3f} mae={val_metrics['mae']:.3f} "
              f"pred_counts={val_metrics['pred_counts']} "
              f"per_class={[round(x, 3) for x in val_metrics['f1_per_class']]}")
        if val_metrics['f1_macro'] > best_val_f1:
            best_val_f1 = val_metrics['f1_macro']
            best_state  = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    # ---- final eval ----
    test_metrics = evaluate(model, test_loader, device, args.head)
    ood_metrics = {}
    for c in ood_corpora:
        loader = make_loader(ood_dfs[c], ood_prompts.get(c), ood_static.get(c), False)
        ood_metrics[c] = evaluate(model, loader, device, args.head)

    # ---- record ----
    config_tag = _config_tag(args)
    rec = {
        'feature_set':  config_tag,
        'classifier':   'hybrid',
        'seed':         args.seed,
        'val_f1_macro': float(best_val_f1),
        'test':         {k: v for k, v in test_metrics.items() if k != 'y_pred'},
        'ood':          {c: {k: v for k, v in m.items() if k != 'y_pred'}
                         for c, m in ood_metrics.items()},
        'fit_seconds':  round(time.time() - t0, 1),
        'flags': {
            'encoder': args.encoder, 'use_scalar_mix': args.use_scalar_mix,
            'use_prompts': args.use_prompts, 'prompt_model': args.prompt_model,
            'use_static': args.use_static, 'head': args.head,
            'use_dann': args.use_dann, 'dann_lambda': args.dann_lambda,
            'dann_warmup_frac': args.dann_warmup_frac,
            'epochs': args.epochs, 'lr': args.lr, 'head_lr': args.head_lr,
            'batch_size': args.batch_size, 'max_len': args.max_len,
            'hidden_dim': args.hidden_dim, 'clip_grad': args.clip_grad,
        },
    }
    _append_result(rec)
    with open(os.path.join(HYBRID_DIR, f'{config_tag}__seed{args.seed}.pkl'), 'wb') as fh:
        pickle.dump({'flags': rec['flags'], 'val': best_val_f1,
                     'test_pred': test_metrics['y_pred'],
                     'ood_pred': {c: m['y_pred'] for c, m in ood_metrics.items()}}, fh)

    ood_str = [f"{c}={m['f1_macro']:.2f}" for c, m in ood_metrics.items()]
    print(f"\n[step8f] {config_tag} seed={args.seed} val={best_val_f1:.3f} "
          f"test={test_metrics['f1_macro']:.3f} qwk={test_metrics['qwk']:.3f} ood={ood_str}")


## Run Configuration

Start with the 1-epoch smoke test. If loss stays finite and prediction counts are not collapsed, switch `--epochs` to `3` and uncomment the variant flags you want.


In [27]:
# Edit this one cell to choose the Step 8f variant.
# Smoke test v1: encoder-only DeBERTa, 1 epoch.
RUN_ARGS = [
    'step8f',
    '--encoder', 'microsoft/deberta-v3-base',
    '--epochs', '1',
    '--batch_size', '16',
    '--max_len', '512',
    '--lr', '2e-5',
    '--head_lr', '1e-4',
    '--clip_grad', '1.0',
]

# Common variants. Uncomment only the lines you need.
# RUN_ARGS += ['--use_scalar_mix']
# RUN_ARGS += ['--use_prompts', '--prompt_model', 'qwen2.5-7b']
# RUN_ARGS += ['--head', 'corn']
# RUN_ARGS += ['--use_dann', '--dann_lambda', '1.0']
# RUN_ARGS += ['--use_static']
# RUN_ARGS[RUN_ARGS.index('1')] = '3'  # full run after smoke test


In [28]:
# Preflight checks before the expensive training cell.
import os, torch, pandas as pd
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA')
required = [
    os.path.join(MULTI_DIR, 'train.csv'),
    os.path.join(MULTI_DIR, 'val.csv'),
    os.path.join(MULTI_DIR, 'test.csv'),
]
missing = [x for x in required if not os.path.exists(x)]
if missing:
    raise FileNotFoundError(missing)
for split in ['train', 'val', 'test']:
    df = pd.read_csv(os.path.join(MULTI_DIR, f'{split}.csv'))
    print(split, 'rows=', len(df), 'labels=', df['education_level'].value_counts().to_dict())
if '--use_prompts' in RUN_ARGS:
    model_key = RUN_ARGS[RUN_ARGS.index('--prompt_model') + 1] if '--prompt_model' in RUN_ARGS else 'qwen2.5-7b'
    for split in ['train', 'val', 'test']:
        path = os.path.join(PROMPT_DIR, f'{split}_prompt_metrics_{model_key}__multi.csv')
        if not os.path.exists(path):
            raise FileNotFoundError(path)
        print(split, 'prompt rows=', len(pd.read_csv(path)))
if '--use_static' in RUN_ARGS:
    for split in ['train', 'val', 'test']:
        path = os.path.join(GENERIC_DIR, f'{split}_static_generic.csv')
        if not os.path.exists(path):
            raise FileNotFoundError(path)
        print(split, 'static rows=', len(pd.read_csv(path)))


GPU: NVIDIA A100-SXM4-40GB
train rows= 6371 labels= {'high': 2400, 'middle': 2107, 'elementary': 1864}
val rows= 795 labels= {'high': 300, 'middle': 263, 'elementary': 232}
test rows= 795 labels= {'high': 300, 'middle': 263, 'elementary': 232}


In [29]:
# Run Step 8f with the selected configuration.
sys.argv = RUN_ARGS
print("Running:", " ".join(sys.argv))
main()


Running: step8f --encoder microsoft/deberta-v3-base --epochs 1 --batch_size 16 --max_len 512 --lr 2e-5 --head_lr 1e-4 --clip_grad 1.0
[step8f] config=deberta-v3-base__softmax seed=42
[step8f] train: rows=6371 labels=[1864, 2107, 2400] sources={'scienceqa': 3600, 'clear': 2771}
[step8f] val: rows=795 labels=[232, 263, 300] sources={'scienceqa': 450, 'clear': 345}
[step8f] test: rows=795 labels=[232, 263, 300] sources={'scienceqa': 450, 'clear': 345}
[step8f] ood_onestop: rows=567 labels=[189, 189, 189] sources={'onestop': 567}
[step8f] ood_openbookqa: rows=1500 labels=[1500, 0, 0] sources={'openbookqa': 1500}
[step8f] ood_race-high: rows=1500 labels=[0, 0, 1500] sources={'race-high': 1500}
[step8f] ood_race-middle: rows=1500 labels=[0, 1500, 0] sources={'race-middle': 1500}
[step8f] sources: {'clear': 0, 'onestop': 1, 'openbookqa': 2, 'race-high': 3, 'race-middle': 4, 'scienceqa': 5}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


train:   0%|          | 0/399 [00:00<?, ?it/s]

[step8f] epoch 1: loss=0.581 (diff=0.581 dom=0.000) val_f1=0.835 qwk=0.877 mae=0.164 pred_counts=[201, 247, 347] per_class=[0.882, 0.745, 0.878]

[step8f] deberta-v3-base__softmax seed=42 val=0.835 test=0.839 qwk=0.880 ood=['onestop=0.24', 'openbookqa=0.11', 'race-high=0.21', 'race-middle=0.12']
